## Clase 7 - Ejercicio: algoritmo de búsqueda para la Torre de Hanoi

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None` si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [ ]:
def search_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    import heapq
    from itertools import count

    # Heurística propia: discos fuera de la varilla meta, más dos movimientos
    # por cada disco pequeño que hoy bloquea la meta al mayor disco desplazado.
    # Los bloqueadores deben salir de la meta y volver; cada disco fuera de la
    # meta debe moverse al menos una vez. Por eso es una cota inferior admisible.
    def heuristic(state):
        disks_outside_goal = state.rods[0] + state.rods[1]
        if not disks_outside_goal:
            return 0
        largest_disk_outside_goal = max(disks_outside_goal)
        blockers = sum(disk < largest_disk_outside_goal for disk in state.rods[2])
        return len(disks_outside_goal) + 2 * blockers

    initial_node = NodeHanoi(initial_state)
    sequence = count()
    frontier = [
        (initial_node.path_cost + heuristic(initial_state), next(sequence), initial_node)
    ]
    best_cost = {initial_state: initial_node.path_cost}
    closed_cost = {}
    visited_states = set()
    nodes_explored = 0
    max_depth = 0
    solution = None

    while frontier:
        _, _, node = heapq.heappop(frontier)
        state = node.state
        cost = node.path_cost

        # Descarta entradas viejas y permite reabrir un estado si aparece
        # después una ruta más barata, necesario para A* con esta heurística.
        if cost != best_cost.get(state):
            continue
        if cost >= closed_cost.get(state, float("inf")):
            continue

        closed_cost[state] = cost
        visited_states.add(state)
        nodes_explored += 1
        max_depth = max(max_depth, node.depth)

        if problem.goal_test(state):
            solution = node
            break

        for child in node.expand(problem):
            child_cost = child.path_cost
            if child_cost < best_cost.get(child.state, float("inf")):
                best_cost[child.state] = child_cost
                priority = child_cost + heuristic(child.state)
                heapq.heappush(frontier, (priority, next(sequence), child))

    nodes_in_frontier = sum(
        1
        for _, _, node in frontier
        if node.path_cost == best_cost.get(node.state)
        and node.path_cost < closed_cost.get(node.state, float("inf"))
    )
    metrics = {
        "solution_found": solution is not None,
        "nodes_explored": nodes_explored,
        "states_visited": len(visited_states),
        "nodes_in_frontier": nodes_in_frontier,
        "max_depth": max_depth,
        "cost_total": float(solution.path_cost) if solution is not None else 0.0,
    }
    #####

    return solution, metrics

Se prueba la función:

In [3]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [4]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: None
states_visited: None
nodes_in_frontier: None
max_depth: None
cost_total: None


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >


Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)